# Building Models

This notebook demonstrates how to build neural network models for PINNs using two approaches:

1. **`create_model`** — high-level shorthand that covers most use cases.
2. **`ModelBase`** — composable layer-by-layer API for full control.

Both approaches produce the same kind of object that plugs directly into the `Trainer`.

## 1. Imports

In [1]:
import jax
import jax.numpy as jnp

# High-level factory
from pinns import create_model

# Low-level building blocks
from pinns.models.model_base import ModelBase
from pinns.models.layers import (
    Normalize, Denormalize,
    FNN, WFFNN, ResNet, PirateNet,
    FourierFeatures,
)

# Domain
from pinns.domain import DomainCubic

## 2. Domain

All models need a domain — it provides input bounds for normalisation, the spatial dimension, and optional time axis.

In [2]:
# 2-D steady-state domain:  x ∈ [0,1],  y ∈ [0,1]
domain_2d = DomainCubic(space=[[0.0, 1.0], [0.0, 1.0]])
domain_2d.set_partition([2, 2])

# 1+1-D transient domain:  x ∈ [0,1],  t ∈ [0,1]
domain_1t = DomainCubic(space=[[0.0, 1.0]], time=[0.0, 1.0])
domain_1t.set_partition([2], time=2)

print("Spatial dims:", domain_2d._spatial_dims)
print("Has time:    ", domain_1t.t_interval)

Spatial dims: 2
Has time:     [0.0, 1.0]


## 3. `create_model` — the shorthand

`create_model(domain, output_dim, ...)` assembles the standard pipeline:

```
Normalize → [features] → Core → [Denormalize]
```

This is the recommended starting point for most problems.

In [3]:
# --- Minimal: FNN with default hidden dims (64, 64, 64) ---
model = create_model(domain_2d, output_dim=1)
print(model)

# --- Custom hidden dims ---
model = create_model(domain_2d, output_dim=1, hidden_dims=(128, 128, 128, 128))
print(model)

# --- Different activation ---
model = create_model(domain_2d, output_dim=1, hidden_dims=(64, 64), activation="swish")
print(model)

# --- With output denormalisation (bounds the output to [0, 1]) ---
model = create_model(
    domain_2d,
    output_dim=1,
    denormalize=True,
    output_range=(0.0, 1.0),
)
print(model)

ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FNN_1] FNN(layer_sizes=[2, 64, 64, 64, 1])
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FNN_1] FNN(layer_sizes=[2, 128, 128, 128, 128, 1])
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FNN_1] FNN(layer_sizes=[2, 64, 64, 1])
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FNN_1] FNN(layer_sizes=[2, 64, 64, 64, 1])
  [Denormalize_2] Denormalize(scaled=True)


### Swap the core network via `create_model`

Pass any layer as `core=` to replace the default `FNN`.

In [4]:
# ResNet core
model_resnet = create_model(
    domain_2d,
    output_dim=1,
    core=ResNet(hidden_dim=64, n_blocks=4),
)
print(model_resnet)

# PirateNet core
model_pirate = create_model(
    domain_2d,
    output_dim=1,
    core=PirateNet(hidden_dim=64, n_blocks=3),
)
print(model_pirate)

# WFFNN core
model_wffnn = create_model(
    domain_2d,
    output_dim=1,
    core=WFFNN([64, 64, 64]),
)
print(model_wffnn)

# FourierFeatures as a features encoder before the FNN core
model_fourier = create_model(
    domain_2d,
    output_dim=1,
    features=FourierFeatures(n_features=64, sigma=1.0,
                              encode_time=False),   # no time dim
    hidden_dims=(64, 64),
)
print(model_fourier)

ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [ResNet_1] ResNet(input_dim=2, hidden_dim=64, n_blocks=4, output_dim=1)
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [PirateNet_1] PirateNet(input_dim=2, hidden_dim=64, n_blocks=3, output_dim=1)
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [WFFNN_1] WFFNN(layer_sizes=[2, 64, 64, 64, 1])
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FourierFeatures_1] FourierFeatures(input_dim=2, n_features=64, sigma=1.0)
  [FNN_2] FNN(layer_sizes=[128, 64, 64, 1])


## 4. `ModelBase` — layer-by-layer

For full control, use `ModelBase` directly. The pipeline you `add()` layers to is built left-to-right. The final layer must reduce the width to `output_dim` — this is done automatically by any core layer (`FNN`, `ResNet`, `PirateNet`, `WFFNN`).

Layer sequence for a basic FNN:
```
Normalize → FNN([64, 64, 64]) → Denormalize
```

In [5]:
key = jax.random.PRNGKey(0)

# --- Single FNN (most common case) ---
# FNN([64, 64, 64]) → layer_sizes = [input_dim, 64, 64, 64, output_dim]
net_fnn = ModelBase(domain_2d, output_dim=1)
net_fnn.add(Normalize())
net_fnn.add(FNN([64, 64, 64]))
print(net_fnn)

params = net_fnn.init(key)
x = jnp.ones((8, 2))
y = net_fnn.apply(params, x)
print("output shape:", y.shape)   # (8, 1)

# --- Stacking two FNNs: [2→64→64→64] → [64→64→64→1] ---
# hidden_dims controls the *intermediate* widths; output_dim adds the final projection.
# To get N total layers, use N-1 entries in hidden_dims.
#
#   FNN([64, 64], output_dim=64)  →  layer_sizes = [2,  64, 64, 64]   (3 layers)
#   FNN([64, 64])                 →  layer_sizes = [64, 64, 64, 1]    (3 layers)
net_stacked = ModelBase(domain_2d, output_dim=1)
net_stacked.add(Normalize())
net_stacked.add(FNN([64, 64], output_dim=64))     # [2 → 64 → 64 → 64]
net_stacked.add(FNN([64, 64]))     # [64 → 64 → 64 → 1]
print(net_stacked)

ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FNN_1] FNN(layer_sizes=[2, 64, 64, 64, 1])


output shape: (8, 1)
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FNN_1] FNN(layer_sizes=[2, 64, 64, 64])
  [FNN_2] FNN(layer_sizes=[64, 64, 64, 1])


In [6]:
# --- ResNet ---
net_resnet = ModelBase(domain_2d, output_dim=1)
net_resnet.add(Normalize())
net_resnet.add(ResNet(hidden_dim=64, n_blocks=4))
print(net_resnet)

# --- PirateNet ---
net_pirate = ModelBase(domain_2d, output_dim=1)
net_pirate.add(Normalize())
net_pirate.add(PirateNet(hidden_dim=64, n_blocks=3))
print(net_pirate)

# --- WFFNN ---
net_wffnn = ModelBase(domain_2d, output_dim=1)
net_wffnn.add(Normalize())
net_wffnn.add(WFFNN([64, 64, 64]))
print(net_wffnn)

# --- FourierFeatures + PirateNet (common combination) ---
net_fourier_pirate = ModelBase(domain_2d, output_dim=1)
net_fourier_pirate.add(Normalize())
net_fourier_pirate.add(FourierFeatures(n_features=64, sigma=1.0,
                                        encode_time=False))
net_fourier_pirate.add(PirateNet(hidden_dim=64, n_blocks=3))
print(net_fourier_pirate)


ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [ResNet_1] ResNet(input_dim=2, hidden_dim=64, n_blocks=4, output_dim=1)
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [PirateNet_1] PirateNet(input_dim=2, hidden_dim=64, n_blocks=3, output_dim=1)
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [WFFNN_1] WFFNN(layer_sizes=[2, 64, 64, 64, 1])
ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [FourierFeatures_1] FourierFeatures(input_dim=2, n_features=64, sigma=1.0)
  [PirateNet_2] PirateNet(input_dim=128, hidden_dim=64, n_blocks=3, output_dim=1)


## 5. Custom layers

Any object that implements three methods can be added to `ModelBase`:

| Method | Purpose |
|---|---|
| `_configure(network, input_dim) -> int` | Called on `add()`. Return the output width. |
| `init(rng) -> dict` | Return trainable parameters (or `{}`). |
| `apply(params, x, params_dict=None) -> array` | Forward pass. |

In [7]:
class SinLayer:
    """Example: parameter-free layer that applies sin to every feature."""

    def _configure(self, network, input_dim: int) -> int:
        return input_dim          # output width = input width (passthrough)

    def init(self, rng) -> dict:
        return {}                 # no trainable parameters

    def apply(self, params, x, params_dict=None):
        return jnp.sin(x)


# Use it as a feature transform before FNN
net_sin = ModelBase(domain_2d, output_dim=1)
net_sin.add(Normalize())
net_sin.add(SinLayer())     # sin of normalised coordinates
net_sin.add(FNN([64, 64]))  # output_dim=None → auto uses network.output_dim=1
print(net_sin)

params = net_sin.init(key)
y = net_sin.apply(params, jnp.ones((4, 2)))
print("output shape:", y.shape)


ModelBase(output_dim=1)
  [Normalize_0] Normalize(n_coord=2, n_context=0, context_scaled=False)
  [SinLayer_1] <__main__.SinLayer object at 0x7f60f79857f0>
  [FNN_2] FNN(layer_sizes=[2, 64, 64, 1])
output shape: (4, 1)


## 6. Domain decomposition — `PartitionFB` / `PartitionX`

Pass a `partition=` strategy to `create_model` (or wrap a `ModelBase` manually) to get a `ModelPartitioned` — a collection of sub-networks, one per subdomain.

| Strategy | Description |
|---|---|
| `PartitionFB` | Fixed-basis partition: subdomains tiled uniformly over the domain. |
| `PartitionX` | X-PINN: arbitrary user-defined subdomains with interface continuity losses. |

In [8]:
from pinns.models.partition import PartitionFB, PartitionX

# --- PartitionFB: 2×2 grid of subdomains over the 2-D domain ---
partition = PartitionFB(overlap=0.1)
PartitionX()

model_fb = create_model(
    domain_2d,
    output_dim=1,
    hidden_dims=(64, 64),
    partition=partition,
)
print(model_fb)
# print("Number of sub-networks:", len(model_fb.networks))

params = model_fb.init(key)
y = model_fb.apply(params, jnp.ones((4, 2)))
y

ModelPartitioned(shape=2×2, strategy=PartitionFB, n_models=4, output_dim=1)
  ├─ sub_0: PartitionFB(overlap=0.1, continuity_weight=1.0, xmin=[0.0, 0.0], xmax=[0.5, 0.5])
  ├─ sub_1: PartitionFB(overlap=0.1, continuity_weight=1.0, xmin=[0.0, 0.5], xmax=[0.5, 1.0])
  ├─ sub_2: PartitionFB(overlap=0.1, continuity_weight=1.0, xmin=[0.5, 0.0], xmax=[1.0, 0.5])
  └─ sub_3: PartitionFB(overlap=0.1, continuity_weight=1.0, xmin=[0.5, 0.5], xmax=[1.0, 1.0])


Array([[0.31656337],
       [0.31656337],
       [0.31656337],
       [0.31656337]], dtype=float32)

## 7. Time-stepping — `stepper=StepperDt(...)`

Pass a stepping strategy to `create_model` to wrap the model with `ModelStepper`. The network receives the solution from the previous time step as additional context columns and steps forward one slice at a time. `partition_time` is automatically set to `False` when a stepper is provided.

| Strategy | Used by | Description |
|---|---|---|
| `StepperDt(dt)` | `ModelStepper.rollout` | Fixed Δt autoregressive rollout |
| `StepperStep(bptt, n_steps)` | Trainer / Problem | Discrete MoL / BPTT; time grid from domain |


In [ ]:
from pinns.models.stepping import StepperDt, StepperStep

model_stepper = create_model(
    domain_1t,            # 1-D spatial + time domain
    output_dim=1,
    hidden_dims=(64, 64),
    stepper=StepperDt(dt=0.1),           # fixed step size for rollout
    context_range=[(-1.0, 1.0)],         # physical range of the u_t context column
)
print(model_stepper)
# print("n_context:", model_stepper.base.n_context)   # automatically set to output_dim=1

ModelStepper(output_dim=1, n_context=1, strategy=StepperDt(dt=0.1), model=ModelBase)


Array([[0.70427823],
       [0.70427823],
       [0.70427823],
       [0.70427823]], dtype=float32)

## Summary

| What you want | How |
|---|---|
| Quick standard FNN | `create_model(domain, output_dim=1)` |
| Different hidden sizes | `create_model(..., hidden_dims=(128,128,128))` |
| Different core (ResNet, PirateNet, WFFNN) | `create_model(..., core=ResNet(64))` |
| Fourier feature encoder | `create_model(..., features=FourierFeatures(...))` |
| Full control / custom layers | `ModelBase(domain, output_dim).add(...).add(...)` |
| Domain decomposition | `create_model(..., partition=PartitionFB(...))` |
| Time-stepping (autoregressive rollout) | `create_model(..., stepper=StepperDt(dt=0.1))` |
| Time-stepping (Trainer / MoL / BPTT) | `create_model(..., stepper=StepperStep(bptt=False))` |

The returned model is passed directly to `Trainer.compile(model=...)`.
